In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import zarr
import tqdm
from torchmetrics import StructuralSimilarityIndexMeasure

In [2]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)

        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        
        # KORREKTUR 1: Passe die Größe von d2 an die von e2 an
        d2 = F.interpolate(d2, size=e2.shape[2:], mode='trilinear', align_corners=False)
        
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)
        
        # KORREKTUR 2: Passe die Größe von d1 an die von e1 an
        d1 = F.interpolate(d1, size=e1.shape[2:], mode='trilinear', align_corners=False)
        
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)

In [3]:
class SupervisedDisplacementDataset(Dataset):
    def __init__(self, moving_path, fixed_path, dvf_gt_path):
        super().__init__()
        self.moving_arr = zarr.open(moving_path, mode='r')
        self.fixed_arr = zarr.open(fixed_path, mode='r')
        self.dvf_gt_arr = zarr.open(dvf_gt_path, mode='r')

        self.num_images = self.moving_arr.shape[3]
        assert self.fixed_arr.shape[3] == self.num_images and self.dvf_gt_arr.shape[3] == self.num_images, \
            "The number of images and DVFs must match across all Zarr arrays."

        self.original_shape = self.moving_arr.shape[:3] # H, W, D
        self.padded_shape = list(self.original_shape)
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
    
    def __len__(self):
        return self.num_images

    def _pad_tensor(self, tensor):
        # Assumes tensor is (C, D, H, W)
        pad_d = self.padded_shape[2] - tensor.shape[1]
        pad_h = self.padded_shape[0] - tensor.shape[2]
        pad_w = self.padded_shape[1] - tensor.shape[3]
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

    def __getitem__(self, idx):
        moving_np = self.moving_arr[..., idx]
        fixed_np = self.fixed_arr[..., idx]
        dvf_gt_np = self.dvf_gt_arr[:, :, :, idx, :] # Shape (H, W, D, 3)

        moving_tensor = torch.from_numpy(moving_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        fixed_tensor = torch.from_numpy(fixed_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        dvf_gt_tensor = torch.from_numpy(dvf_gt_np.astype(np.float32)).permute(3, 2, 0, 1)

        moving_padded = self._pad_tensor(moving_tensor)
        fixed_padded = self._pad_tensor(fixed_tensor)
        dvf_gt_padded = self._pad_tensor(dvf_gt_tensor)

        return moving_padded, fixed_padded, dvf_gt_padded

In [4]:
class NCCLoss(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        pred_flat = pred.view(pred.size(0), -1)
        target_flat = target.view(target.size(0), -1)

        pred_mean = pred_flat.mean(dim=1, keepdim=True)
        target_mean = target_flat.mean(dim=1, keepdim=True)

        pred_std = pred_flat.std(dim=1, keepdim=True)
        target_std = target_flat.std(dim=1, keepdim=True)

        pred_centered = pred_flat - pred_mean
        target_centered = target_flat - target_mean
        
        cross_corr = (pred_centered * target_centered).sum(dim=1, keepdim=True)
        
        ncc = cross_corr / (pred_std * target_std * pred_flat.size(1) + self.eps)
        
        return 1 - ncc.mean() # Return 1 - NCC so lower is better

In [5]:
class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super().__init__()
        vectors = [torch.arange(0, s) for s in size]
        grids = torch.meshgrid(vectors, indexing='ij')
        grid = torch.stack(grids).float()
        self.register_buffer('grid', grid.unsqueeze(0), persistent=False)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        new_locs = new_locs.permute(0, 2, 3, 4, 1)
        new_locs = new_locs[..., [2, 1, 0]]
        return F.grid_sample(src, new_locs, align_corners=True, padding_mode="border")

In [22]:
MOVING_PATH = "../DCE_codeset/MRI-Datasets/DCE"
FIXED_PATH = "../DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
DVF_GT_PATH = "../DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/transfo_zarr_2.zarr"
MODEL_PATH = "dvf_model_01_1e-5_200ep.pth"

In [2]:
indices_load_path = 'test_indices2.pth'
BATCH_SIZE = 2

full_dataset = SupervisedDisplacementDataset(MOVING_PATH, FIXED_PATH, DVF_GT_PATH)

loaded_test_indices = torch.load(indices_load_path)
print(f"Gespeicherte Test-Indizes aus '{indices_load_path}' geladen.")

test_dataset = Subset(full_dataset, loaded_test_indices)
print(f"Test-Set mit {len(test_dataset)} Proben wurde rekonstruiert.")

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("\nDer DataLoader für das reproduzierbare Test-Set ist bereit.")

NameError: name 'SupervisedDisplacementDataset' is not defined

In [8]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')   
padded_shape_for_stn = (full_dataset.padded_shape[2], full_dataset.padded_shape[0], full_dataset.padded_shape[1])

model = UNet3D().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

spatial_transformer = SpatialTransformer3D(size=padded_shape_for_stn).to(device)

print(f"Modell von {MODEL_PATH} geladen. Evaluiere auf {len(test_dataset)} Testbildern.")

Modell von dvf_model_01_1e-5_200ep.pth geladen. Evaluiere auf 200 Testbildern.


In [9]:
mse_fn = nn.MSELoss()
ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device) 
ncc_loss_fn = NCCLoss()

total_mse = 0.0
total_ssim = 0.0
total_ncc_loss = 0.0

/home/shooty/anaconda3/envs/ds-env-01/lib/python3.13/site-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `StructuralSimilarityIndexMeasure` from `torchmetrics` was deprecated and will be removed in 2.0. Import `StructuralSimilarityIndexMeasure` from `torchmetrics.image` instead.
  _future_warning(


In [10]:
with torch.no_grad():
    progress_bar = tqdm.tqdm(test_loader, desc="Evaluiere Testset")
    for moving_batch, fixed_batch, dvf_gt_batch in progress_bar:
        moving_batch = moving_batch.to(device)
        fixed_batch = fixed_batch.to(device)

        predicted_dvf = model(fixed_batch, moving_batch)
        
        warped_batch = spatial_transformer(moving_batch, predicted_dvf)

        mse = mse_fn(warped_batch, fixed_batch)
        ssim = ssim_fn(warped_batch, fixed_batch)
        ncc_loss = ncc_loss_fn(warped_batch, fixed_batch)

        total_mse += mse.item()
        total_ssim += ssim.item()
        total_ncc_loss += ncc_loss.item()

avg_mse = total_mse / len(test_loader)
avg_ssim = total_ssim / len(test_loader)
avg_ncc_loss = total_ncc_loss / len(test_loader)

print("\n--- Evaluationsergebnisse ---")
print(f"Durchschnittlicher MSE: {avg_mse:.6f}")
print(f"Durchschnittlicher SSIM: {avg_ssim:.6f}")
print(f"Durchschnittlicher NCC-Loss (1 - NCC): {avg_ncc_loss:.6f}")

Evaluiere Testset: 100%|██████████| 100/100 [01:22<00:00,  1.21it/s]


--- Evaluationsergebnisse ---
Durchschnittlicher MSE: 576.317655
Durchschnittlicher SSIM: 0.986019
Durchschnittlicher NCC-Loss (1 - NCC): 0.002234


In [11]:
import shutil
results_path = 'MRI-Datasets/mdreg_motion_correction_results'
if os.path.exists(results_path):
    shutil.rmtree(results_path) # Lösche alte Ergebnisse für dieses Beispiel
os.makedirs(results_path)

In [3]:
OUTPUT_WARPED_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/inference_warped_images.zarr"
OUTPUT_DVF_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/inference_predicted_dvfs.zarr"

In [4]:
moving_batch.shape

NameError: name 'moving_batch' is not defined

In [16]:
fixed_batch.shape

torch.Size([2, 1, 52, 256, 256])

In [ ]:
num_test_images = len(test_dataset)
sample_warped_shape = test_dataset[0][0].shape # (C, D, H, W) for one padded image
sample_dvf_shape = test_dataset[0][2].shape   # (C, D, H, W) for one padded DVF

output_warped_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w', 
                               shape=(num_test_images,) + sample_warped_shape,
                               chunks=(1,) + sample_warped_shape, # Save one image per chunk
                               dtype='float32')
                               
output_dvf_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w',
                            shape=(num_test_images,) + sample_dvf_shape,
                            chunks=(1,) + sample_dvf_shape,
                            dtype='float32')

# --- 5. Run Inference and Save Loop ---
current_index = 0
with torch.no_grad():
    for moving_batch, fixed_batch, _ in progress_bar: # dvf_gt is not needed
        moving_batch = moving_batch.to(device)
        fixed_batch = fixed_batch.to(device)
        # Predict the displacement field (DVF)
        predicted_dvf = model(moving_batch[], moving_batch)
        
        # Apply the DVF to the moving image to get the corrected/warped image
        warped_batch = spatial_transformer(moving_batch, predicted_dvf)
        
        # Move results to CPU and convert to NumPy
        warped_np = warped_batch.cpu().numpy()
        dvf_np = predicted_dvf.cpu().numpy()
        
        # Write the batch of results to the Zarr files
        batch_size_current = warped_np.shape[0]
        output_warped_zarr[current_index : current_index + batch_size_current] = warped_np
        output_dvf_zarr[current_index : current_index + batch_size_current] = dvf_np
        
        current_index += batch_size_current

Verarbeite Testset: 100%|██████████| 100/100 [01:35<00:00,  1.05it/s]


In [4]:
if __name__ == '__main__':
    # --- 1. Konfiguration ---
    INPUT_ZARR_PATH = '/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/MRI-Datasets/DCE'
    MODEL_PATH = '/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/dvf_model_01_1e-5_100ep.pth'
    OUTPUT_WARPED_PATH = 'warped_images.zarr'
    OUTPUT_DVF_PATH = 'displacement_fields.zarr'
    FIXED_SLICE_INDEX = 50


In [6]:

    # --- 2. Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Lade das Zarr-Array (lazy)
    input_zarr = zarr.open(INPUT_ZARR_PATH, mode='r')
    H, W, D, T = input_zarr.shape
    print(f"Input data shape: (H={H}, W={W}, D={D}, T={T})")

    # --- 3. Modell laden ---
    # Annahme: Modell wurde mit gepaddeten Daten trainiert. Padding wird hier nicht gezeigt,
    # da wir das Modell direkt auf die Originalgröße anwenden.
    # Die input_size für den SpatialTransformer muss (Tiefe, Höhe, Breite) sein.
    model_input_size = (D, H, W)
    
    model = UNet3D().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    
    spatial_transformer = SpatialTransformer3D(size=model_input_size).to(device)
    print("Model loaded successfully.")
    output_warped_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w',
                                   shape=input_zarr.shape,
                                   chunks=input_zarr.chunks,
                                   dtype=input_zarr.dtype)

    # Die Displacement Fields haben eine zusätzliche Dimension für die 3 Vektorkomponenten
    dvf_shape = (H, W, D, T, 3)
    # Passende Chunks für die DVF-Ausgabe
    dvf_chunks = input_zarr.chunks + (3,) if input_zarr.chunks else None
    output_dvf_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w',
                                shape=dvf_shape,
                                chunks=dvf_chunks,
                                dtype='float32')

    print(f"Output for warped images created at: {OUTPUT_WARPED_PATH}")
    print(f"Output for DVF created at: {OUTPUT_DVF_PATH}")

    # --- 5. Fixes Bild vorbereiten ---
    def preprocess_volume(volume_np):
        # Konvertiere (H, W, D) NumPy-Array zu (B=1, C=1, D, H, W) PyTorch-Tensor
        tensor = torch.from_numpy(volume_np.astype(np.float32)).permute(2, 0, 1) # (D, H, W)
        tensor = tensor.unsqueeze(0).unsqueeze(0) # (B=1, C=1, D, H, W)
        return tensor.to(device)

    print(f"Preparing fixed image from time slice {FIXED_SLICE_INDEX}...")
    fixed_np = input_zarr[..., FIXED_SLICE_INDEX]
    fixed_tensor = preprocess_volume(fixed_np)
    print("Fixed image is ready on device.")

    # --- 6. Inferenz-Schleife ---
    with torch.no_grad():
        for i in tqdm.trange(T, desc="Processing time slices"):
            # Lade das aktuelle 'moving' Bild
            moving_np = input_zarr[..., i]
            moving_tensor = preprocess_volume(moving_np)

            # Führe die Inferenz durch
            predicted_dvf = model(fixed_tensor, moving_tensor)
            
            # Wende die Transformation an
            warped_image = spatial_transformer(moving_tensor, predicted_dvf)

            # --- Nachbearbeitung und Speichern ---
            # Konvertiere Tensoren zurück zu NumPy-Arrays in der Speicherreihenfolge (H, W, D)
            warped_np = warped_image.squeeze().cpu().numpy().transpose(1, 2, 0)
            dvf_np = predicted_dvf.squeeze().cpu().numpy().transpose(2, 3, 1, 0)
            
            # Speichere die Ergebnisse im Zarr-Array
            output_warped_zarr[..., i] = warped_np
            output_dvf_zarr[..., i, :] = dvf_np

    print("\n--- Processing complete ---")
    print(f"Warped images saved to: {OUTPUT_WARPED_PATH}")
    print(f"Displacement fields saved to: {OUTPUT_DVF_PATH}")

Using device: cuda
Input data shape: (H=256, W=256, D=50, T=1000)


NameError: name 'UNet3D' is not defined

In [5]:
import helpers
import cupy as cp

In [6]:
output_dvf_zarr = zarr.open(OUTPUT_DVF_PATH, mode='r')

In [7]:
output_warped_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='r')

In [8]:
input_zarr = zarr.open(INPUT_ZARR_PATH, mode='r')

In [9]:
slice = 25

In [10]:
fixed_zarr = zarr.open(FIXED_PATH, mode='r')
fixed_zarr.shape

(256, 256, 50, 1000)

In [11]:
output = output_warped_zarr[:,:,slice]
output = cp.transpose(output, [2,1,0])
raw = input_zarr[:,:,slice]
raw = cp.transpose(raw, [2,1,0])
fixed = fixed_zarr[:,:,slice]
fixed = cp.transpose(fixed, [2,1,0])

In [22]:
helpers.explore_3D_array_comparison_and_diff(fixed, output, 'twilight')

interactive(children=(IntSlider(value=500, description='Slice:', max=999), Output()), _dom_classes=('widget-in…

In [23]:
import mdreg

In [ ]:
plot = mdreg.plot.series(input_zarr, fixed_zarr, output_warped_zarr)

In [14]:
import napari
import numpy as np

In [15]:
# Erstelle deine zwei Dummy-Datensätze (wie im Beispiel oben)
shape = (60, 256, 256); dataset1 = np.zeros(shape, dtype=np.uint8); dataset2 = np.zeros(shape, dtype=np.uint8)
for z in range(shape[0]):
    y, x = np.ogrid[-128:128, -128:128]; radius = 50; center_y = int(30 * np.sin(z * 2 * np.pi / shape[0])); mask = x*x + (y-center_y)*(y-center_y) <= radius*radius; dataset1[z, mask] = 255
dataset2[:, 80:180, 80:180] = 255

# Starte den Napari-Viewer
viewer = napari.Viewer()

# Füge beide Datensätze als separate Ebenen hinzu
viewer.add_image(dataset1, name='Dataset 1')
viewer.add_image(dataset2, name='Dataset 2')

# Zeige die Ebenen nebeneinander an, indem du den Grid-Modus aktivierst
viewer.grid.enabled = True

napari.run()

: 

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import imageio
import os
import shutil

# --- 1. Vorbereitung: Lade oder erstelle deine zwei Datensätze ---
# Annahme: Beide Arrays haben die gleiche Form (Tiefe, Höhe, Breite)
# Erstelle Dummy-Daten für dieses Beispiel
shape = (60, 256, 256) # 60 Schichten, 256x256 Pixel
dataset1 = np.zeros(shape, dtype=np.uint8)
dataset2 = np.zeros(shape, dtype=np.uint8)

# Zeichne eine Form in die Dummy-Daten, um sie zu unterscheiden
# Ein sich bewegender Kreis in Dataset 1
for z in range(shape[0]):
    y, x = np.ogrid[-128:128, -128:128]
    radius = 50
    center_y = int(30 * np.sin(z * 2 * np.pi / shape[0]))
    mask = x*x + (y-center_y)*(y-center_y) <= radius*radius
    dataset1[z, mask] = 255

# Ein statisches Quadrat in Dataset 2
dataset2[:, 80:180, 80:180] = 255

# --- 2. Setup für die Videoerstellung ---
frames_dir = "temp_frames"
video_path = "vergleichsvideo.mp4"
fps = 15 # Bilder pro Sekunde

# Erstelle einen temporären Ordner für die Bilder
if os.path.exists(frames_dir):
    shutil.rmtree(frames_dir)
os.makedirs(frames_dir)

print(f"Erstelle {shape[0]} Bilder im Ordner '{frames_dir}'...")

# --- 3. Erstelle für jeden Slice ein Bild ---
for i in range(shape[0]):
    # Erstelle eine Abbildung mit zwei Plots nebeneinander
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Plotte das erste Dataset
    axes[0].imshow(dataset1[i, :, :], cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'Dataset 1 (Slice {i+1}/{shape[0]})')
    axes[0].axis('off')
    
    # Plotte das zweite Dataset
    axes[1].imshow(dataset2[i, :, :], cmap='gray', vmin=0, vmax=255)
    axes[1].set_title(f'Dataset 2 (Slice {i+1}/{shape[0]})')
    axes[1].axis('off')
    
    plt.tight_layout()
    
    # Speichere das Bild in den temporären Ordner
    # Das Format %04d sorgt für eine korrekte Sortierung (z.B. 0001, 0002, ...)
    frame_path = os.path.join(frames_dir, f'frame_{i:04d}.png')
    plt.savefig(frame_path)
    plt.close(fig) # Schließe die Abbildung, um Speicher zu sparen

print("Alle Bilder erfolgreich erstellt.")

# --- 4. Füge die Bilder zu einem Video zusammen ---
print(f"Erstelle Video '{video_path}' mit {fps} FPS...")
# imageio's "with"-Syntax stellt sicher, dass alles korrekt geschlossen wird
with imageio.get_writer(video_path, fps=fps) as writer:
    # Lese alle erstellten Bilder in sortierter Reihenfolge und füge sie zum Video hinzu
    for filename in sorted(os.listdir(frames_dir)):
        if filename.endswith('.png'):
            image_path = os.path.join(frames_dir, filename)
            frame = imageio.imread(image_path)
            writer.append_data(frame)

# --- 5. Aufräumen ---
print("Räume temporären Ordner auf...")
shutil.rmtree(frames_dir)

print(f"✔️ Video erfolgreich erstellt und unter '{video_path}' gespeichert.")

Erstelle 60 Bilder im Ordner 'temp_frames'...


/tmp/ipykernel_1824/272067592.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imageio.imread(image_path)
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1200, 600) to (1200, 608) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Alle Bilder erfolgreich erstellt.
Erstelle Video 'vergleichsvideo.mp4' mit 15 FPS...
Räume temporären Ordner auf...
✔️ Video erfolgreich erstellt und unter 'vergleichsvideo.mp4' gespeichert.


In [27]:
import zarr
import numpy as np
import imageio
import matplotlib.pyplot as plt
import tqdm
import os
import shutil

# =================================================================================
# 1. KONFIGURATION
# =================================================================================

# --- BITTE DIESE PFADE ANPASSEN ---
# Annahme: Deine Daten haben die Form (Höhe, Breite, Tiefe, Zeit) -> (256, 256, 50, 1000)

VIDEO_OUTPUT_PATH = 'comparison_video_slice_25.mp4'
WARPED_ZARR_PATH = OUTPUT_WARPED_PATH
INPUT_ZARR_PATH = INPUT_ZARR_PATH
# --- VIDEO-EINSTELLUNGEN ---
FPS = 8
# KORREKTUR: Lege die zu zeigende Tiefenschicht fest (26. Schicht = Index 25)
DEPTH_SLICE_TO_SHOW = 26

# =================================================================================
# --- Optional: Dummy-Daten erstellen, falls die Pfade nicht existieren ---
def create_dummy_data_if_needed(path, shape=(256, 256, 50, 1000)):
    if not os.path.exists(path):
        print(f"Dummy-Daten nicht unter '{path}' gefunden. Erstelle sie zur Demonstration.")
        data = np.zeros(shape, dtype=np.uint8)
        zarr.save(path, data)

create_dummy_data_if_needed(INPUT_ZARR_PATH)
create_dummy_data_if_needed(WARPED_ZARR_PATH)
# =================================================================================

# 2. ZARR-ARRAYS LADEN
print("Lade Zarr-Arrays...")
input_arr = zarr.open(INPUT_ZARR_PATH, mode='r')
warped_arr = zarr.open(WARPED_ZARR_PATH, mode='r')

assert input_arr.shape == warped_arr.shape, "Array-Formen stimmen nicht überein!"

H, W, D, T = input_arr.shape
print(f"Daten geladen. Form: (H={H}, W={W}, D={D}, T={T})")

# 3. VIDEO ERSTELLEN
print(f"Erstelle Video '{VIDEO_OUTPUT_PATH}' mit {FPS} FPS...")

with imageio.get_writer(VIDEO_OUTPUT_PATH, fps=FPS) as writer:
    for i in tqdm.trange(T, desc="Generiere Videoframes"):
        slice_input = input_arr[:, :, DEPTH_SLICE_TO_SHOW, i]
        slice_warped = warped_arr[:, :, DEPTH_SLICE_TO_SHOW, i]
        diff_slice = np.abs(slice_input.astype(float) - slice_warped.astype(float))

        rotated_input = np.rot90(slice_input)
        rotated_warped = np.rot90(slice_warped)
        rotated_diff = np.rot90(diff_slice)
        rotated_input = np.rot90(rotated_input)
        rotated_warped = np.rot90(rotated_warped)
        rotated_diff = np.rot90(rotated_diff)
        rotated_input = np.rot90(rotated_input)
        rotated_warped = np.rot90(rotated_warped)
        rotated_diff = np.rot90(rotated_diff)
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f'Time: {i+1}/{T}  |  Depth Slice: {DEPTH_SLICE_TO_SHOW + 1}', fontsize=16)

        axes[0].imshow(rotated_input, cmap='gray')
        axes[0].set_title('Input Dataset')
        axes[0].axis('off')

        axes[1].imshow(rotated_warped, cmap='gray')
        axes[1].set_title('Warped Dataset')
        axes[1].axis('off')

        # --- NEUER PLOT: Zeige das Differenzbild an ---
        # Eine andere Colormap (wie 'inferno') hebt Unterschiede gut hervor
        axes[2].imshow(rotated_diff, cmap='inferno')
        axes[2].set_title('Difference')
        axes[2].axis('off')
        
        plt.tight_layout(rect=[0, 0, 1, 0.95])

        fig.canvas.draw()
        frame_rgba = np.asarray(fig.canvas.buffer_rgba())
        frame_rgb = frame_rgba[:, :, :3]
        writer.append_data(frame_rgb)
        plt.close(fig)


print(f"\n✔️ Video erfolgreich erstellt unter: {VIDEO_OUTPUT_PATH}")

Lade Zarr-Arrays...
Daten geladen. Form: (H=256, W=256, D=50, T=1000)
Erstelle Video 'comparison_video_slice_25.mp4' mit 8 FPS...


Generiere Videoframes:   0%|          | 0/1000 [00:00<?, ?it/s]

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1800, 600) to (1808, 608) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
Generiere Videoframes: 100%|██████████| 1000/1000 [02:18<00:00,  7.23it/s]



✔️ Video erfolgreich erstellt unter: comparison_video_slice_25.mp4
